In [ ]:
!pip install ultralytics supervision opencv-python numpy pandas -q

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import supervision as sv
from collections import defaultdict, deque
from google.colab import files
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
import base64

print("✅ Libraries Installed & Ready.")

In [ ]:
def draw_zone(image_path, zone_name):
  frame = cv2.imread(image_path)
  h, w = frame.shape[:2]
  scale = 800 / w
  resized = cv2.resize(frame, (int(w*scale), int(h*scale)))
  _, buf = cv2.imencode('.jpg', resized)
  img_data = 'data:image/jpeg;base64,' + base64.b64encode(buf).decode('utf-8')

  js = Javascript('''
    async function draw(img_src, zone_name) {
      const div = document.createElement('div');
      div.innerHTML = `<h3>Draw ${zone_name} (Click corners, Double-click to finish)</h3><canvas id="c"></canvas><br><button id="btn">Save & Continue</button>`;
      document.body.append(div);
      const canvas = document.getElementById('c');
      const ctx = canvas.getContext('2d');
      const img = new Image(); img.src = img_src;
      await new Promise(r => img.onload = r);
      canvas.width = img.width; canvas.height = img.height;
      ctx.drawImage(img,0,0);
      let pts = [];
      canvas.onclick = e => {
        const r = canvas.getBoundingClientRect();
        pts.push([Math.round((e.clientX-r.left)), Math.round((e.clientY-r.top))]);
        ctx.fillStyle='lime'; ctx.fillRect(pts[pts.length-1][0]-2, pts[pts.length-1][1]-2, 4, 4);
        if (pts.length>1) {
            ctx.beginPath(); ctx.moveTo(pts[pts.length-2][0], pts[pts.length-2][1]);
            ctx.lineTo(pts[pts.length-1][0], pts[pts.length-1][1]); ctx.strokeStyle='lime'; ctx.stroke();
        }
      };
      canvas.ondblclick = () => {
        ctx.beginPath(); ctx.moveTo(pts[pts.length-1][0], pts[pts.length-1][1]);
        ctx.lineTo(pts[0][0], pts[0][1]); ctx.stroke();
      };
      await new Promise(r => document.getElementById('btn').onclick = () => { div.remove(); r(); });
      return pts;
    }
  ''')
  display(js)
  pts = eval_js(f'draw("{img_data}", "{zone_name}")')
  return (np.array(pts) / scale).astype(int)

# --- EXECUTE DRAWING ---
VIDEO_PATH = "abc.mp4" # Ensure this matches your file name!
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
if ret:
    cv2.imwrite("ref.jpg", frame)
    print("👉 Step 1: Draw the FLOOR (Walking Area)")
    FLOOR_POLY = draw_zone("ref.jpg", "FLOOR ZONE")
    
    print("👉 Step 2: Draw SHELF A (First Aisle)")
    SHELF_A_POLY = draw_zone("ref.jpg", "SHELF A")

    print("👉 Step 3: Draw SHELF B (Second Aisle) - Double Click if none.")
    SHELF_B_POLY = draw_zone("ref.jpg", "SHELF B")
    
    print("✅ Zones Saved.")
else:
    print(f"❌ Error: Could not read {VIDEO_PATH}. Upload it first.")

In [ ]:
# --- CONFIGURATION ---
OUTPUT_VIDEO = "final_insight_output.mp4"
OUTPUT_CSV = "customer_journey_report.csv"
WRIST_CROP = 130 
CONSENSUS_FRAMES = 1  # MUST see product for 5 frames to flag "PICKED"
RETURN_FRAMES = 4     # MUST see empty hand for 8 frames to flag "RETURNED"

# COCO Classes relevant to a Sports Store (Strict Filter)
# 24:Backpack, 25:Umbrella, 26:Handbag, 28:Suitcase, 32:Ball, 34:Bat, 36:Skateboard, 
# 38:Tennis Racket, 39:Bottle, 41:Cup, 43:Knife, 44:Spoon (Camping), etc.
VALID_ITEMS = [24, 25, 26, 28, 32, 34, 35, 36, 38, 39, 41, 43, 44, 45]

class Customer:
    def __init__(self, id):
        self.id = id
        self.entry_time = 0
        self.exit_time = 0
        self.state = "WALKING" # States: WALKING, BROWSING, PICKED, RETURNED
        self.current_zone = "FLOOR"
        self.products_handled = [] # List of products touched
        self.journey = []          # List of zones visited
        self.pick_buffer = 0       # Counter for "Pick" confirmation
        self.return_buffer = 0     # Counter for "Return" confirmation
        self.history = deque(maxlen=30) # For drawing trail

def main():
    # 1. LOAD MODELS
    # Using 'Large' Pose for skeleton accuracy, 'Medium' Object for balance
    print("⏳ Loading AI Models (ByteTrack Enabled)...")
    pose_model = YOLO('yolov8l-pose.pt') 
    obj_model = YOLO('yolov8m.pt')

    cap = cv2.VideoCapture(VIDEO_PATH)
    W, H = int(cap.get(3)), int(cap.get(4))
    fps = int(cap.get(5))
    out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    # Zones Setup
    zones = {
        "SHELF_A": sv.PolygonZone(SHELF_A_POLY, triggering_anchors=[sv.Position.CENTER]),
        "SHELF_B": sv.PolygonZone(SHELF_B_POLY, triggering_anchors=[sv.Position.CENTER])
    }
    floor_poly = sv.PolygonZone(FLOOR_POLY)

    customers = {}
    frame_n = 0
    print(f"🚀 Starting Advanced Analysis ({W}x{H})...")

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        frame_n += 1
        curr_time = frame_n / fps
        
        # --- A. TRACKING (ByteTrack for better ID stability) ---
        # tracker="bytetrack.yaml" helps prevent ID switching when they turn around
        results = pose_model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
        
        if results[0].boxes.id is not None:
            ids = results[0].boxes.id.int().cpu().numpy()
            boxes = results[0].boxes.xyxy.cpu().numpy()
            kpts = results[0].keypoints.xy.cpu().numpy()

            for i, tid in enumerate(ids):
                # Init Customer
                if tid not in customers: 
                    customers[tid] = Customer(tid)
                    customers[tid].entry_time = curr_time
                
                c = customers[tid]
                c.exit_time = curr_time
                box = boxes[i]
                
                # Update Movement History
                center = (int((box[0]+box[2])/2), int(box[3])) # Feet position
                c.history.append(center)

                # --- B. ZONE DETECTION ---
                # Check Keypoints 9 (Left Wrist) & 10 (Right Wrist)
                lw, rw = kpts[i][9], kpts[i][10]
                active_shelf = None
                wrist_pt = None
                
                # Check Shelf Interaction
                for z_name, z_poly in [(n, zones[n].polygon) for n in zones]:
                    if cv2.pointPolygonTest(z_poly, (lw[0],lw[1]), False) >= 0: 
                        active_shelf, wrist_pt = z_name, lw
                        break
                    elif cv2.pointPolygonTest(z_poly, (rw[0],rw[1]), False) >= 0: 
                        active_shelf, wrist_pt = z_name, rw
                        break
                
                # Update Journey Logic
                current_loc = active_shelf if active_shelf else "FLOOR"
                if c.current_zone != current_loc:
                    c.current_zone = current_loc
                    if current_loc not in c.journey: c.journey.append(current_loc)

                # --- C. BEHAVIOR LOGIC (State Machine) ---
                
                # Default Color: Green (Walking)
                color = (0, 255, 0)
                
                if active_shelf:
                    # HE IS INTERACTING
                    
                    # 1. Product Detection Logic
                    # Only check every 3 frames to save GPU
                    detected_item = None
                    if frame_n % 3 == 0:
                        x, y = int(wrist_pt[0]), int(wrist_pt[1])
                        x1, x2 = max(0, x-WRIST_CROP), min(W, x+WRIST_CROP)
                        y1, y2 = max(0, y-WRIST_CROP), min(H, y+WRIST_CROP)
                        crop = frame[y1:y2, x1:x2]
                        
                        if crop.size > 0:
                            # Run Detection on Hand Crop
                            res = obj_model(crop, verbose=False, conf=0.15)
                            for b in res[0].boxes:
                                cls_id = int(b.cls[0])
                                if cls_id in VALID_ITEMS:
                                    detected_item = res[0].names[cls_id]
                                    # Visual Debug of Crop
                                    cv2.rectangle(frame, (x1,y1), (x2,y2), (255,0,255), 1)
                                    break
                    
                    # 2. State Transitions
                    if detected_item:
                        c.pick_buffer += 1
                        c.return_buffer = 0 # Reset return counter
                    else:
                        c.pick_buffer = max(0, c.pick_buffer - 1)
                        # Only count towards return if they previously took something
                        if c.state == "PICKED": c.return_buffer += 1

                    # 3. Confirm State Changes
                    if c.pick_buffer >= CONSENSUS_FRAMES:
                        c.state = "PICKED"
                        if detected_item not in c.products_handled:
                            c.products_handled.append(detected_item)
                    
                    elif c.return_buffer >= RETURN_FRAMES and c.state == "PICKED":
                        c.state = "RETURNED"

                    # 4. Shelf Colors
                    if c.state == "PICKED": color = (0, 0, 255)     # Red
                    elif c.state == "RETURNED": color = (255, 255, 0) # Cyan/Yellow
                    else: 
                        c.state = "BROWSING"
                        color = (0, 165, 255) # Orange

                else:
                    # NOT INTERACTING (Walking on Floor)
                    if c.state == "BROWSING": c.state = "WALKING" # Reset if just looking
                    # If PICKED, they stay PICKED (Red)
                    if c.state == "PICKED": color = (0, 0, 255)
                    # If RETURNED, they go back to Walking (Green) or stay Returned logic?
                    # Let's keep them flagged as "Returned" for the visual history
                    if c.state == "RETURNED": color = (255, 255, 0)

                # --- D. VISUALIZATION ---
                # 1. Bounding Box
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 2)
                
                # 2. Label
                label = f"ID:{tid} {c.state}"
                if c.products_handled: label += f" [{c.products_handled[-1]}]"
                cv2.putText(frame, label, (int(box[0]), int(box[1])-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                
                # 3. Draw Trail (Movement Heatmap)
                if len(c.history) > 2:
                    points = np.array(c.history, dtype=np.int32).reshape((-1, 1, 2))
                    cv2.polylines(frame, [points], False, color, 2)

        # Draw Zones
        cv2.polylines(frame, [FLOOR_POLY], True, (50, 50, 50), 1)
        cv2.polylines(frame, [SHELF_A_POLY, SHELF_B_POLY], True, (255, 255, 0), 2)
        
        # Draw Dashboard
        cv2.rectangle(frame, (0,0), (400, 60), (0,0,0), -1)
        cv2.putText(frame, f"Active Customers: {len(ids) if 'ids' in locals() else 0}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(frame, "Red=Took | Cyan=Returned | Orange=Browsing", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)

        out.write(frame)
        if frame_n % 10 == 0: print(f"Processing {curr_time:.1f}s...", end='\r')

    cap.release()
    out.release()

    # --- GENERATE SALES INSIGHTS REPORT ---
    rows = []
    for c in customers.values():
        # Determine Final Conversion Status
        conversion = "Browsed Only"
        if c.state == "PICKED": conversion = "Converted (Picked)"
        elif c.state == "RETURNED": conversion = "Interested but Returned"
        
        journey_str = " -> ".join(c.journey) if c.journey else "Floor Only"
        product_str = ", ".join(c.products_handled) if c.products_handled else "None"
        
        rows.append({
            "Customer_ID": c.id,
            "Total_Dwell_Time": round(c.exit_time - c.entry_time, 2),
            "Final_Status": c.state,
            "Conversion_Type": conversion,
            "Products_Interacted": product_str,
            "Store_Journey_Path": journey_str
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)
    
    print("\n✅ Analysis Complete. Downloading Report & Video...")
    files.download(OUTPUT_VIDEO)
    files.download(OUTPUT_CSV)

if __name__ == "__main__":
    if 'SHELF_A_POLY' in globals(): main()
    else: print("❌ Error: Please Run Step 2 (Drawing) First!")